# SE-801 Lecture 9: PyTorch Engineering and Reproducible Experiments

This notebook inspects module state, verifies reproducible DataLoader order,
checks weight reloads, and proves exact CPU continuation from a checkpoint.

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

SEED = 20261102
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
REFERENCE_DEVICE = torch.device("cpu")
print("PyTorch:", torch.__version__)
print("Available device:", DEVICE)
print("Exact-resume reference device:", REFERENCE_DEVICE)

PyTorch: 2.14.0+cu130
Available device: cuda
Exact-resume reference device: cpu


## 1. Data, split, and train-only preprocessing

In [2]:
def make_split(per_class: int, *, seed: int):
    generator = torch.Generator().manual_seed(seed)
    centers = torch.tensor([[-1.2, -0.7, 0.8, 0.0], [1.0, -0.8, -0.4, 0.9], [0.1, 1.2, 0.3, -0.9]])
    features, targets = [], []
    for class_id, center in enumerate(centers):
        x = center + 0.55 * torch.randn(per_class, 4, generator=generator)
        x[:, 3] += 0.25 * x[:, 0] * x[:, 1]
        features.append(x)
        targets.append(torch.full((per_class,), class_id, dtype=torch.long))
    x, y = torch.cat(features), torch.cat(targets)
    order = torch.randperm(len(y), generator=generator)
    return x[order], y[order]


raw_train_x, train_y = make_split(180, seed=SEED + 1)
raw_val_x, val_y = make_split(60, seed=SEED + 2)
raw_test_x, test_y = make_split(60, seed=SEED + 3)
mean = raw_train_x.mean(0)
std = raw_train_x.std(0).clamp_min(1e-6)
train_x = (raw_train_x - mean) / std
val_x = (raw_val_x - mean) / std
test_x = (raw_test_x - mean) / std

## 2. Custom modules and registered state

In [3]:
class ResidualMLPBlock(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))

    def forward(self, x):
        return torch.relu(x + self.net(x))


class ResearchMLP(nn.Module):
    def __init__(self, in_features=4, hidden=24, classes=3, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_features, hidden), nn.ReLU())
        self.block = ResidualMLPBlock(hidden)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, classes)
        self.register_buffer("input_mean", torch.zeros(in_features))

    def forward(self, x):
        z = self.encoder(x - self.input_mean)
        z = self.block(z)
        return self.head(self.dropout(z))


torch.manual_seed(SEED + 10)
inspection_model = ResearchMLP()
parameter_shapes = {name: list(parameter.shape) for name, parameter in inspection_model.named_parameters()}
state_shapes = {name: list(tensor.shape) for name, tensor in inspection_model.state_dict().items()}
parameter_count = sum(p.numel() for p in inspection_model.parameters() if p.requires_grad)
print("Trainable parameters:", parameter_count)
print("State keys:")
for key, shape in state_shapes.items():
    print(" ", key, shape)

Trainable parameters: 1395
State keys:
  input_mean [4]
  encoder.0.weight [24, 4]
  encoder.0.bias [24]
  block.net.0.weight [24, 24]
  block.net.0.bias [24]
  block.net.2.weight [24, 24]
  block.net.2.bias [24]
  head.weight [3, 24]
  head.bias [3]


## 3. Reproducible loader order

A small dataset returns its own row index. Fresh generators with equal state
produce equal shuffled orders. Advancing one generator changes future orders.

In [4]:
class IndexedDataset(Dataset):
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index], index


def loader_order(generator):
    loader = DataLoader(IndexedDataset(train_x, train_y), batch_size=64, shuffle=True, generator=generator, num_workers=0)
    return [int(index) for _, _, indices in loader for index in indices]


gen_a = torch.Generator().manual_seed(SEED + 20)
gen_b = torch.Generator().manual_seed(SEED + 20)
first_order_a = loader_order(gen_a)
first_order_b = loader_order(gen_b)
second_order_a = loader_order(gen_a)
loader_first_order_equal = first_order_a == first_order_b
advanced_order_differs = second_order_a != first_order_b
print("Fresh seeded loader orders equal:", loader_first_order_equal)
print("Advanced generator changes the next order:", advanced_order_differs)

Fresh seeded loader orders equal: True
Advanced generator changes the next order: True


## 4. State-dict reload and prediction equivalence

In [5]:
def build_model(seed: int):
    torch.manual_seed(seed)
    model = ResearchMLP()
    with torch.no_grad():
        model.input_mean.copy_(torch.zeros_like(model.input_mean))
    return model


source_model = build_model(SEED + 30).to(REFERENCE_DEVICE)
source_model.eval()
frozen_batch = test_x[:24].to(REFERENCE_DEVICE)
with torch.no_grad():
    source_logits = source_model(frozen_batch)

weights_path = OUTPUT_DIR / "model_state_dict.pt"
torch.save(source_model.state_dict(), weights_path)
reloaded_model = build_model(SEED + 31).to(REFERENCE_DEVICE)
reloaded_state = torch.load(weights_path, map_location=REFERENCE_DEVICE, weights_only=True)
load_result = reloaded_model.load_state_dict(reloaded_state, strict=True)
reloaded_model.eval()
with torch.no_grad():
    reloaded_logits = reloaded_model(frozen_batch)
reload_prediction_max_error = float((source_logits - reloaded_logits).abs().max())
print("Strict load result:", load_result)
print("Reload prediction max error:", reload_prediction_max_error)

Strict load result: <All keys matched successfully>
Reload prediction max error: 0.0


## 5. Exact CPU checkpoint continuation

The reference run uses dropout and a shuffled DataLoader. Exact continuation
therefore restores model, optimizer, loader-generator, and CPU RNG state.

In [6]:
def accuracy(logits, targets):
    return float((logits.argmax(1) == targets).float().mean())


def make_training_state():
    model = build_model(SEED + 40).to(REFERENCE_DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loader_generator = torch.Generator().manual_seed(SEED + 50)
    return model, optimizer, loader_generator


def train_epoch(model, optimizer, loader_generator, epoch):
    loader = DataLoader(
        TensorDataset(train_x, train_y), batch_size=60, shuffle=True,
        generator=loader_generator, num_workers=0,
    )
    model.train()
    for x, y in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        train_logits = model(train_x)
        val_logits = model(val_x)
    return {
        "epoch": epoch,
        "train_loss": float(F.cross_entropy(train_logits, train_y)),
        "validation_loss": float(F.cross_entropy(val_logits, val_y)),
        "train_accuracy": accuracy(train_logits, train_y),
        "validation_accuracy": accuracy(val_logits, val_y),
    }


def uninterrupted_run():
    torch.manual_seed(SEED + 60)
    model, optimizer, loader_generator = make_training_state()
    history = [train_epoch(model, optimizer, loader_generator, epoch) for epoch in range(1, 11)]
    return model, history


def interrupted_run():
    torch.manual_seed(SEED + 60)
    model, optimizer, loader_generator = make_training_state()
    history = [train_epoch(model, optimizer, loader_generator, epoch) for epoch in range(1, 6)]
    checkpoint_path = OUTPUT_DIR / "resume_checkpoint.pt"
    checkpoint = {
        "epoch": 5,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loader_generator_state": loader_generator.get_state(),
        "cpu_rng_state": torch.get_rng_state(),
        "config": {"hidden": 24, "dropout": 0.15, "batch_size": 60, "lr": 0.01},
    }
    torch.save(checkpoint, checkpoint_path)

    restored_model = build_model(SEED + 999).to(REFERENCE_DEVICE)
    restored_optimizer = torch.optim.Adam(restored_model.parameters(), lr=0.01)
    restored_generator = torch.Generator()
    loaded = torch.load(checkpoint_path, map_location=REFERENCE_DEVICE, weights_only=True)
    restored_model.load_state_dict(loaded["model_state_dict"], strict=True)
    restored_optimizer.load_state_dict(loaded["optimizer_state_dict"])
    restored_generator.set_state(loaded["loader_generator_state"])
    torch.set_rng_state(loaded["cpu_rng_state"])
    history.extend(
        train_epoch(restored_model, restored_optimizer, restored_generator, epoch)
        for epoch in range(loaded["epoch"] + 1, 11)
    )
    return restored_model, history, checkpoint_path


full_model, full_history = uninterrupted_run()
resumed_model, resumed_history, checkpoint_path = interrupted_run()
full_model.eval()
resumed_model.eval()
with torch.no_grad():
    full_logits = full_model(test_x)
    resumed_logits = resumed_model(test_x)
resume_prediction_max_error = float((full_logits - resumed_logits).abs().max())
resume_state_max_error = max(
    float((full_model.state_dict()[key] - resumed_model.state_dict()[key]).abs().max())
    for key in full_model.state_dict()
)
history_max_error = max(
    abs(a[field] - b[field])
    for a, b in zip(full_history, resumed_history)
    for field in ["train_loss", "validation_loss", "train_accuracy", "validation_accuracy"]
)
test_accuracy = accuracy(full_logits, test_y)
print("Resume prediction max error:", resume_prediction_max_error)
print("Resume state max error:", resume_state_max_error)
print("History max error:", history_max_error)
print("Final test accuracy:", test_accuracy)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
epochs = [row["epoch"] for row in full_history]
axes[0].plot(epochs, [row["validation_loss"] for row in full_history], label="uninterrupted", color="#246B8E", linewidth=2.2)
axes[0].plot(epochs, [row["validation_loss"] for row in resumed_history], "--", label="resumed", color="#B87521")
axes[1].plot(epochs, [row["validation_accuracy"] for row in full_history], label="uninterrupted", color="#246B8E", linewidth=2.2)
axes[1].plot(epochs, [row["validation_accuracy"] for row in resumed_history], "--", label="resumed", color="#B87521")
axes[0].set(title="Validation loss", xlabel="epoch", ylabel="cross-entropy")
axes[1].set(title="Validation accuracy", xlabel="epoch", ylabel="accuracy", ylim=(0.3, 1.02))
for axis in axes:
    axis.axvline(5, color="#60717E", linewidth=1, alpha=0.7)
    axis.grid(alpha=0.25)
    axis.legend()
fig.savefig(OUTPUT_DIR / "resume_equivalence_curves.png", dpi=170)
plt.close(fig)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()

Resume prediction max error: 0.0
Resume state max error: 0.0
History max error: 0.0
Final test accuracy: 1.0


## 6. Save the registered evidence

In [7]:
results = {
    "seed": SEED,
    "reference_device": str(REFERENCE_DEVICE),
    "available_device": str(DEVICE),
    "protocol": {
        "task": "three-class synthetic four-feature classification",
        "split_sizes": {"train": len(train_y), "validation": len(val_y), "test": len(test_y)},
        "architecture": "ResearchMLP with one residual block and dropout",
        "optimizer": "Adam",
        "epochs": 10,
        "batch_size": 60,
        "num_workers": 0,
    },
    "module": {
        "parameter_count": parameter_count,
        "parameter_shapes": parameter_shapes,
        "state_shapes": state_shapes,
    },
    "loader": {
        "fresh_seeded_orders_equal": loader_first_order_equal,
        "advanced_generator_changes_order": advanced_order_differs,
        "first_twelve_indices": first_order_a[:12],
    },
    "reload": {
        "prediction_max_error": reload_prediction_max_error,
        "weights_sha256": sha256(weights_path),
        "weights_bytes": weights_path.stat().st_size,
    },
    "resume": {
        "prediction_max_error": resume_prediction_max_error,
        "state_max_error": resume_state_max_error,
        "history_max_error": history_max_error,
        "test_accuracy": test_accuracy,
        "uninterrupted_history": full_history,
        "resumed_history": resumed_history,
        "checkpoint_sha256": sha256(checkpoint_path),
        "checkpoint_bytes": checkpoint_path.stat().st_size,
    },
}
(OUTPUT_DIR / "experiment_results.json").write_text(json.dumps(results, indent=2))
print("Wrote", OUTPUT_DIR / "experiment_results.json")

Wrote outputs/experiment_results.json
